# Amazon ML — Entity Matching Pipeline (Colab)

Runs the whole pipeline end to end, in order:

1. **Normalize** — clean names/addresses -> `normalized/`
2. **Blocking** — generate candidate pairs -> `candidates/`
3. **Training pairs** — label candidates (positives + hard/easy negatives) -> `pairs/`
4. **Feature engineering** — name/address/country/component/cross-field features -> `features/`
5. **Validation set** — held-out S1 entities scored on ALL their candidates (test-like)
6. **Train** LightGBM, plus XGBoost / logistic regression / ExtraTrees to compare against
7. **Pick the best model and threshold** on macro-F0.5 (the challenge metric), then add **exclusive assignment** (each S2/S3 record goes to one S1 entity) and an **error analysis**
8. **Predict the test set** and write `output/matching_results.tsv` + `output/candidate_pairs.tsv`
9. **Validate** the submission with the official validator

**Before running:** the dataset must be in your Google Drive
(`MyDrive/amazon_ml/6ab10eb3b23ba_student_resource/...` unzipped, or the `.zip` in `MyDrive/amazon_ml/`).
Then run the cells top to bottom.

> Set `MODE = "smoke"` (first 5000 rows, free tier, checks the code runs) or `"full"` (real run — needs a
> **High-RAM** runtime). Smoke mode has almost no true matches, so its scores are meaningless.

## 0. Setup: mount Drive, unzip dataset, define paths

In [ ]:
import os
import sys
import zipfile

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- EDIT THESE IF YOUR DRIVE LAYOUT DIFFERS ----
DRIVE_ROOT = "/content/drive/MyDrive/amazon_ml"
WORK_DIR = "/content/data"   # only used if we have to unzip
# -------------------------------------------------

REL = "6ab10eb3b23ba_student_resource/student_resource"
DRIVE_DATASET = os.path.join(DRIVE_ROOT, REL, "dataset/")
LOCAL_DATASET = os.path.join(WORK_DIR, REL, "dataset/")
ZIP_PATH = os.path.join(DRIVE_ROOT, "6ab10eb3b23ba_student_resource.zip")

if os.path.isdir(os.path.join(DRIVE_DATASET, "train")):
    DATASET_DIR = DRIVE_DATASET                       # already unzipped in Drive
    print("Using dataset already unzipped in Drive:", DATASET_DIR)
elif os.path.isdir(os.path.join(LOCAL_DATASET, "train")):
    DATASET_DIR = LOCAL_DATASET
    print("Using local dataset:", DATASET_DIR)
elif os.path.isfile(ZIP_PATH):
    print(f"Unzipping {ZIP_PATH} -> {WORK_DIR} (a few minutes for ~3 GB)...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(WORK_DIR)
    DATASET_DIR = LOCAL_DATASET
else:
    raise FileNotFoundError(f"Neither {DRIVE_DATASET} nor {ZIP_PATH} exists -- fix DRIVE_ROOT above.")

RAW_TRAIN_DIR = os.path.join(DATASET_DIR, "train/")
RAW_TEST_DIR = os.path.join(DATASET_DIR, "test/")
NORM_DIR = os.path.join(DATASET_DIR, "normalized/")
CAND_DIR = os.path.join(DATASET_DIR, "candidates/")
PAIRS_DIR = os.path.join(DATASET_DIR, "pairs/")
FEATURES_DIR = os.path.join(DATASET_DIR, "features/")
STUDENT_RESOURCE_DIR = os.path.dirname(DATASET_DIR.rstrip("/"))
OUTPUT_DIR = os.path.join(STUDENT_RESOURCE_DIR, "output/")      # matching_results.tsv + candidate_pairs.tsv
MODEL_DIR = os.path.join(STUDENT_RESOURCE_DIR, "model/")
VALIDATOR = os.path.join(STUDENT_RESOURCE_DIR, "utils", "validate_submission.py")

for d in (NORM_DIR, CAND_DIR, PAIRS_DIR, FEATURES_DIR, OUTPUT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

## 1. Normalize (`normalize.py`)

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import json
import unicodedata
# Standardize abbreviations (Add more as needed)
NAME_REPLACEMENTS = {
    r'\binc\b': 'incorporated',
    r'\bcorp\b': 'corporation',
    r'\bltd\b': 'limited',
    r'\bpvt\b': 'private',
    r'\bllc\b': 'limited liability company',
    r'\bco\b': 'company'
}

ADDRESS_REPLACEMENTS = {
    r'\bst\b': 'street',
    r'\brd\b': 'road',
    r'\bave\b': 'avenue',
    r'\bblvd\b': 'boulevard',
    r'\bdr\b': 'drive',
    r'\bln\b': 'lane',
    r'\bct\b': 'court',
    r'\bapt\b': 'apartment',
    r'\bste\b': 'suite',
    r'\bhwy\b': 'highway'
}

# Multi-word phrases must precede their component words: regex alternation is
# first-match, not longest-match, so "limited" would otherwise match first and
# strand "liability company" behind (e.g. "acme llc" -> "acme  liability ").
LEGAL_SUFFIXES_TO_REMOVE = r'\b(limited liability company|incorporated|corporation|limited|private|company)\b'

# Bump this to force every normalized file to be regenerated on the next run (outputs are tagged with the
# version that produced them, so files from an older normalizer are never silently reused).
#   "2" = Indic vowel signs kept (see _PUNCT) + legal suffixes stripped longest-phrase-first
NORMALIZE_VERSION = "2"
_VERSION_FILE = os.path.join(NORM_DIR, ".normalize_version.json")

# Strip punctuation/symbols but KEEP letters, digits, whitespace and Indic scripts (U+0900-0DFF: Devanagari ...
# Malayalam). Plain [^\w\s] also deletes the combining vowel signs (matras), which turned "राम" into "र म".
# The Devanagari danda/abbreviation marks (U+0964, U+0965, U+0970) are punctuation, so they are still removed.
_PUNCT = re.compile(r"[^\w\s\u0900-\u0DFF]|[\u0964\u0965\u0970]")

In [ ]:
def clean_text(text, replacements):
    """
    Lowercases, normalizes unicode, handles '&', and standardizes.
    """
    if pd.isna(text):
        return ""
    
    # Lowercase
    text = str(text).lower()
    
    # Unicode normalization (e.g., handles French accents well without destroying Hindi characters)
    text = unicodedata.normalize('NFKC', text)
    
    # Explicitly handle '&' vs 'and' before removing punctuation
    text = text.replace('&', ' and ')
    
    # Apply standard abbreviation replacements
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)
        
    # Remove punctuation except spaces (keep unicode characters for Hindi/French)
    text = _PUNCT.sub(' ', text)
    
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def remove_legal_suffixes(text):
    """Removes expanded legal suffixes from the normalized text."""
    text = re.sub(LEGAL_SUFFIXES_TO_REMOVE, '', text)
    return re.sub(r'\s+', ' ', text).strip()

def normalize_dataframe(df):
    """
    Normalizes the names and addresses in the dataframe, maintaining original representations.
    """
    print(f"Normalizing dataframe with {len(df)} rows...")
    
    # 1. Handle missing values (keeping original columns untouched mostly, just filling NaN)
    df['business_name'] = df['business_name'].fillna("")
    df['business_address'] = df['business_address'].fillna("")
    
    # 2. Clean Text (creates new columns)
    print("Cleaning business names...")
    df['norm_name'] = df['business_name'].apply(lambda x: clean_text(x, NAME_REPLACEMENTS))
    
    print("Creating name without legal suffixes...")
    df['norm_name_no_suffix'] = df['norm_name'].apply(remove_legal_suffixes)
    
    print("Cleaning business addresses...")
    df['norm_address'] = df['business_address'].apply(lambda x: clean_text(x, ADDRESS_REPLACEMENTS))
    
    # Note: Tokens and character representations will be handled dynamically during feature extraction
    # to save disk space in the TSV, but the original text is preserved!
    return df

def _done_versions():
    try:
        with open(_VERSION_FILE) as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def process_and_save(filename, source_dir):
    """Loads, normalizes, and saves the dataframe. Skips only files made by the CURRENT normalizer version."""
    print(f"\n--- Processing {filename} ---")
    out_filepath = os.path.join(NORM_DIR, filename)
    done = _done_versions()
    if os.path.exists(out_filepath):
        if done.get(filename) == NORMALIZE_VERSION:
            print(f"File {out_filepath} is up to date (normalizer v{NORMALIZE_VERSION}). Skipping.")
            return
        print(f"File {out_filepath} was made by an older normalizer -> regenerating.")

    filepath = os.path.join(source_dir, filename)

    # Load dataset
    df = pd.read_csv(filepath, sep="\t")

    # Normalize
    df_norm = normalize_dataframe(df)

    # Save to output directory (write-then-rename: a crash never leaves a half-written file that looks valid)
    tmp_path = out_filepath + ".tmp"
    df_norm.to_csv(tmp_path, sep="\t", index=False)
    os.replace(tmp_path, out_filepath)
    done[filename] = NORMALIZE_VERSION
    with open(_VERSION_FILE, "w") as f:
        json.dump(done, f)
    print(f"Saved normalized data to {out_filepath}")


In [ ]:
def main():
    print("Starting Normalization Process...")
    
    # Train files
    train_files = ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv"]
    for file in train_files:
        process_and_save(file, RAW_TRAIN_DIR)
        
    # Test files
    test_files = ["test_source1.tsv", "test_source2.tsv", "test_source3.tsv"]
    for file in test_files:
        process_and_save(file, RAW_TEST_DIR)

    print("\nNormalization Complete! All files saved in", NORM_DIR)

main()

## 2. Blocking (`blocking.py`)

Five blocking strategies (name prefix, name tokens, address tokens, TF-IDF name,
TF-IDF address) unioned and capped per Source-1 entity.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
import gc
import os
import time
# Only the columns blocking actually needs -- cuts memory roughly in half on the
# multi-GB normalized files (skips the raw business_name/business_address and
# the suffix-bearing norm_name column, which nothing here reads).
BLOCKING_COLUMNS = ["entity_id", "country", "norm_name_no_suffix", "norm_address"]

# ------------------------------------------------------------------
# Tunables. Frequency caps are expressed as a FRACTION of the target
# source's size, not a fixed count -- a fixed count (e.g. 300) means a
# vastly looser filter once you go from a 50k-row smoke sample to the
# full 5M-row source, which is exactly what blew up the candidate sets
# in the original script (median ~464 candidates/entity on a 50k-row
# sample against a 50k-row target -- i.e. ~1% of the entire target pool
# per entity, not a "block").
# ------------------------------------------------------------------
FREQ_CAP_FRACTION = 0.0008   # a blocking key/token used by more than this
                             # fraction of the target pool is too generic
MIN_FREQ_CAP = 30            # floor so small samples/countries aren't gutted

MIN_TOKEN_LEN_NAME = 4
MIN_TOKEN_LEN_ADDR_ALPHA = 5
MIN_TOKEN_LEN_ADDR_NUM = 3
MIN_OVERLAP_ADDR = 2         # address words (street/road/city) are individually
                             # common; require >=2 shared tokens to count as a
                             # block match. Names are short, so 1 shared rare
                             # token is kept for name-token blocking.

MAX_CANDIDATES_PER_ENTITY = 75   # hard cap on what candidate_pairs.tsv can hold
                                 # per Source-1 entity -- this is what actually
                                 # gets fed to the matching model, so an
                                 # unbounded union here just moves the O(N*M)
                                 # blow-up one stage downstream instead of
                                 # fixing it.

MIN_CANDIDATES_BEFORE_TFIDF = 3  # TF-IDF blocks only run for S1 entities that
                                  # the cheap blocks (prefix/token) didn't
                                  # already cover well -- this is the single
                                  # biggest lever for making this runnable on
                                  # a laptop: TF-IDF cost scales with the
                                  # number of queries, and most entities don't
                                  # need it once names/addresses roughly match.

TFIDF_TOP_K = 8
TFIDF_DIST_THRESHOLD = 0.6

In [ ]:
def _freq_cap(target_len):
    return max(MIN_FREQ_CAP, int(target_len * FREQ_CAP_FRACTION))


def _load_source(prefix, name, sample_size):
    path = os.path.join(NORM_DIR, f"{prefix}_{name}.tsv")
    df = pd.read_csv(path, sep="\t", usecols=BLOCKING_COLUMNS, nrows=sample_size)
    df["norm_name_no_suffix"] = df["norm_name_no_suffix"].fillna("")
    df["norm_address"] = df["norm_address"].fillna("")
    df["country"] = df["country"].astype(str)
    return df

In [ ]:
# ============================================================
# BLOCK 1: Name Prefix Blocking (Country + first N chars)
# ============================================================
def block_name_prefix(s1, s_target, prefix_lengths=(3, 4, 5)):
    """Multiple name prefix blocks to catch variations."""
    all_pairs = []
    for n in prefix_lengths:
        # Filter on the actual name length (not the combined "country_prefix"
        # key length -- that conflated the country label's length with the
        # prefix length and filtered inconsistently across countries).
        s1_ok = s1["norm_name_no_suffix"].str.len() >= n
        t_ok = s_target["norm_name_no_suffix"].str.len() >= n

        s1_key = s1.loc[s1_ok, "country"] + "_" + s1.loc[s1_ok, "norm_name_no_suffix"].str[:n]
        t_key = s_target.loc[t_ok, "country"] + "_" + s_target.loc[t_ok, "norm_name_no_suffix"].str[:n]

        s1_temp = pd.DataFrame({"entity_id_s1": s1.loc[s1_ok, "entity_id"], "bk": s1_key})
        t_temp = pd.DataFrame({"entity_id_cand": s_target.loc[t_ok, "entity_id"], "bk": t_key})

        cap = _freq_cap(len(s_target))
        t_counts = t_temp["bk"].value_counts()
        valid_keys = t_counts[t_counts <= cap].index
        t_temp = t_temp[t_temp["bk"].isin(valid_keys)]

        merged = pd.merge(s1_temp, t_temp, on="bk")[["entity_id_s1", "entity_id_cand"]]
        all_pairs.append(merged)

    return pd.concat(all_pairs, ignore_index=True).drop_duplicates() if all_pairs else pd.DataFrame(columns=["entity_id_s1", "entity_id_cand"])

In [ ]:
# ============================================================
# Shared vectorized token-overlap blocking (names or addresses)
# ============================================================
def _token_overlap_pairs(s1, s_target, id_col_s1, id_col_t, text_col, extract_tokens, min_overlap):
    """Explode text into blocking tokens, cap over-common ones, join on shared
    tokens, and require >= min_overlap distinct shared tokens per pair. This
    replaces the original per-row Python loops (iterrows over millions of rows
    with a hand-rolled dict index) with vectorized pandas operations, and adds
    the overlap requirement that turns a "any shared word" match (extremely
    weak at this data scale) into a real signal.
    """
    s1_tok = pd.DataFrame({
        "entity_id_s1": s1[id_col_s1],
        "country": s1["country"],
        "token": s1[text_col].map(extract_tokens),
    }).explode("token").dropna(subset=["token"])
    s1_tok["bk"] = s1_tok["country"] + "_" + s1_tok["token"]

    t_tok = pd.DataFrame({
        "entity_id_cand": s_target[id_col_t],
        "country": s_target["country"],
        "token": s_target[text_col].map(extract_tokens),
    }).explode("token").dropna(subset=["token"])
    t_tok["bk"] = t_tok["country"] + "_" + t_tok["token"]

    cap = _freq_cap(len(s_target))
    t_counts = t_tok["bk"].value_counts()
    valid_keys = t_counts[t_counts <= cap].index
    t_tok = t_tok[t_tok["bk"].isin(valid_keys)]

    merged = pd.merge(s1_tok[["entity_id_s1", "bk"]], t_tok[["entity_id_cand", "bk"]], on="bk")
    del s1_tok, t_tok
    gc.collect()

    if merged.empty:
        return pd.DataFrame(columns=["entity_id_s1", "entity_id_cand"])

    overlap = merged.groupby(["entity_id_s1", "entity_id_cand"]).size()
    pairs = overlap[overlap >= min_overlap].index.to_frame(index=False)
    return pairs


def _name_tokens(text):
    toks = text.split()
    return [t for t in toks if len(t) >= MIN_TOKEN_LEN_NAME] or None


def _addr_tokens(text):
    toks = text.split()
    out = [t for t in toks if (t.isdigit() and len(t) >= MIN_TOKEN_LEN_ADDR_NUM) or (len(t) >= MIN_TOKEN_LEN_ADDR_ALPHA and t.isalpha())]
    return out or None


# ============================================================
# BLOCK 2: Name Token Overlap Blocking
# ============================================================
def block_name_tokens(s1, s_target):
    return _token_overlap_pairs(s1, s_target, "entity_id", "entity_id", "norm_name_no_suffix", _name_tokens, min_overlap=1)


# ============================================================
# BLOCK 3: Address Token Overlap Blocking
# ============================================================
def block_address_tokens(s1, s_target):
    return _token_overlap_pairs(s1, s_target, "entity_id", "entity_id", "norm_address", _addr_tokens, min_overlap=MIN_OVERLAP_ADDR)

In [ ]:
# ============================================================
# BLOCK 4/5: TF-IDF Blocking (char n-grams, per country)
# ============================================================
def block_tfidf(s1, s_target, text_column, top_k=TFIDF_TOP_K, batch_size=10000, distance_threshold=TFIDF_DIST_THRESHOLD):
    """TF-IDF char n-gram blocking, done PER COUNTRY to enforce country match.
    `s1` is expected to already be filtered down to the entities that need
    this fallback -- see generate_all_candidates."""
    if len(s1) == 0:
        return pd.DataFrame(columns=["entity_id_s1", "entity_id_cand"])

    countries = set(s1["country"].unique()) & set(s_target["country"].unique())

    all_pairs = []
    for country in countries:
        s1_c = s1[s1["country"] == country].reset_index(drop=True)
        t_c = s_target[s_target["country"] == country].reset_index(drop=True)

        if len(s1_c) == 0 or len(t_c) == 0:
            continue

        print(f"    TF-IDF [{text_column}] country={country}: {len(s1_c)} queries vs {len(t_c)} targets")
        s1_texts = s1_c[text_column].tolist()
        t_texts = t_c[text_column].tolist()

        vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), min_df=2, max_df=0.9,
                                     max_features=50000, dtype=np.float32)
        t_matrix = vectorizer.fit_transform(t_texts)
        s1_matrix = vectorizer.transform(s1_texts)

        actual_k = min(top_k, t_matrix.shape[0])
        nn = NearestNeighbors(n_neighbors=actual_k, metric="cosine", n_jobs=-1)
        nn.fit(t_matrix)

        s1_ids = s1_c["entity_id"].values
        t_ids = t_c["entity_id"].values

        for start in range(0, s1_matrix.shape[0], batch_size):
            end = min(start + batch_size, s1_matrix.shape[0])
            dists, idxs = nn.kneighbors(s1_matrix[start:end])

            for i in range(end - start):
                for j in range(actual_k):
                    if dists[i, j] < distance_threshold:
                        all_pairs.append({
                            "entity_id_s1": s1_ids[start + i],
                            "entity_id_cand": t_ids[idxs[i, j]],
                        })

        del s1_matrix, t_matrix, vectorizer, nn
        gc.collect()

    return pd.DataFrame(all_pairs).drop_duplicates() if all_pairs else pd.DataFrame(columns=["entity_id_s1", "entity_id_cand"])


def block_tfidf_address(s1, s_target, **kwargs):
    """TF-IDF on addresses PER COUNTRY. Catches transliteration cases where
    names share zero character overlap but addresses overlap significantly."""
    return block_tfidf(s1, s_target, text_column="norm_address", **kwargs)

In [ ]:
# ============================================================
# MAIN PIPELINE
# ============================================================
def _cap_per_entity(tagged_blocks, max_candidates=MAX_CANDIDATES_PER_ENTITY):
    """Union candidates across blocks, scored by how many distinct blocking
    strategies agreed on each pair ("support"), and keep only the top
    `max_candidates` per Source-1 entity. Without this, a single noisy block
    (or several together) can hand hundreds of candidates per entity to the
    matching stage -- this is the hard ceiling on candidate_pairs.tsv size."""
    frames = [df.assign(block=name) for name, df in tagged_blocks if len(df) > 0]
    if not frames:
        return pd.DataFrame(columns=["entity_id_s1", "entity_id_cand"])

    combined = pd.concat(frames, ignore_index=True)
    del frames
    support = combined.groupby(["entity_id_s1", "entity_id_cand"])["block"].nunique().reset_index(name="support")
    del combined
    gc.collect()

    support = support.sort_values(["entity_id_s1", "support"], ascending=[True, False])
    capped = support.groupby("entity_id_s1", sort=False).head(max_candidates)
    return capped[["entity_id_s1", "entity_id_cand"]]


def generate_all_candidates(s1, s_target, source_label, tfidf_mode="fallback"):
    """Union of all blocking strategies for S1 -> one target source.
    tfidf_mode: "fallback" (default) runs TF-IDF only for S1 entities the
    cheap blocks didn't already cover -- much cheaper at this data scale.
    "full" runs TF-IDF for every S1 entity (only feasible with real compute
    budget, e.g. a cloud box, not a laptop)."""
    print(f"\n  === Blocking S1 -> {source_label} ({len(s1)} x {len(s_target)}) ===")

    t0 = time.time()
    print("  [1/5] Name Prefix Blocking...")
    b1 = block_name_prefix(s1, s_target)
    print(f"         -> {len(b1)} pairs ({time.time()-t0:.1f}s)")

    t0 = time.time()
    print("  [2/5] Name Token Blocking...")
    b2 = block_name_tokens(s1, s_target)
    print(f"         -> {len(b2)} pairs ({time.time()-t0:.1f}s)")

    t0 = time.time()
    print("  [3/5] Address Token Blocking...")
    b3 = block_address_tokens(s1, s_target)
    print(f"         -> {len(b3)} pairs ({time.time()-t0:.1f}s)")

    if tfidf_mode == "full":
        s1_for_tfidf = s1
    else:
        cheap = pd.concat([b1, b2, b3], ignore_index=True).drop_duplicates()
        cheap_counts = cheap.groupby("entity_id_s1").size()
        del cheap
        well_covered = set(cheap_counts[cheap_counts >= MIN_CANDIDATES_BEFORE_TFIDF].index)
        s1_for_tfidf = s1[~s1["entity_id"].isin(well_covered)]
        print(f"  TF-IDF fallback: {len(s1_for_tfidf)}/{len(s1)} entities need it "
              f"({len(s1) - len(s1_for_tfidf)} already had >= {MIN_CANDIDATES_BEFORE_TFIDF} cheap candidates)")

    t0 = time.time()
    print("  [4/5] TF-IDF Name Blocking (per country)...")
    b4 = block_tfidf(s1_for_tfidf, s_target, text_column="norm_name_no_suffix")
    print(f"         -> {len(b4)} pairs ({time.time()-t0:.1f}s)")

    t0 = time.time()
    print("  [5/5] TF-IDF Address Blocking (per country)...")
    b5 = block_tfidf_address(s1_for_tfidf, s_target)
    print(f"         -> {len(b5)} pairs ({time.time()-t0:.1f}s)")

    capped = _cap_per_entity([
        ("name_prefix", b1), ("name_tokens", b2), ("address_tokens", b3),
        ("tfidf_name", b4), ("tfidf_address", b5),
    ])
    print(f"  Capped union (<= {MAX_CANDIDATES_PER_ENTITY}/entity): {len(capped)} pairs")

    del b1, b2, b3, b4, b5
    gc.collect()

    return capped

In [ ]:
def _candidate_file(prefix, sample_size):
    suffix = "_TEST" if sample_size else ""
    return os.path.join(CAND_DIR, f"{prefix}_candidate_pairs{suffix}.tsv")


def print_candidate_stats(df):
    counts = df["candidate_entity_ids"].fillna("").apply(lambda x: 0 if x == "" else len(x.split(",")))
    print(f"  entities: {len(df)} | zero-candidate: {(counts==0).sum()} | "
          f"mean: {counts.mean():.1f} | median: {counts.median():.0f} | max: {counts.max()}")


def run_blocking_pipeline(is_test=False, sample_size=None, tfidf_mode="fallback"):
    prefix = "test" if is_test else "train"
    print(f"\n{'='*60}\n  BLOCKING PIPELINE — {prefix.upper()} DATA\n{'='*60}")
    if sample_size:
        print(f"  SMOKE-TEST MODE: only first {sample_size} rows of each source")

    print("Loading normalized data...")
    s1 = _load_source(prefix, "source1", sample_size)
    s2 = _load_source(prefix, "source2", sample_size)
    s3 = _load_source(prefix, "source3", sample_size)

    s2_candidates = generate_all_candidates(s1, s2, "Source 2", tfidf_mode=tfidf_mode)
    del s2
    gc.collect()

    s3_candidates = generate_all_candidates(s1, s3, "Source 3", tfidf_mode=tfidf_mode)
    del s3
    gc.collect()

    print("\nUnioning S2 + S3 candidates and formatting output...")
    final_candidates = pd.concat([s2_candidates, s3_candidates], ignore_index=True).drop_duplicates()
    del s2_candidates, s3_candidates
    gc.collect()

    grouped = final_candidates.groupby("entity_id_s1")["entity_id_cand"].apply(
        lambda x: ",".join(sorted(set(x)))
    ).reset_index()
    grouped.rename(columns={"entity_id_s1": "source1_entity_id", "entity_id_cand": "candidate_entity_ids"}, inplace=True)

    all_s1_ids = pd.DataFrame({"source1_entity_id": s1["entity_id"]})
    final_output = pd.merge(all_s1_ids, grouped, on="source1_entity_id", how="left")
    final_output["candidate_entity_ids"] = final_output["candidate_entity_ids"].fillna("")

    out_file = _candidate_file(prefix, sample_size)
    final_output.to_csv(out_file, sep="\t", index=False)
    print(f"\nSUCCESS! Candidate pairs saved to {out_file}")
    print_candidate_stats(final_output)

In [ ]:
def evaluate_recall(sample_size=None):
    """Measure recall ceiling against ground truth (train only -- test has no labels)."""
    print("\n" + "=" * 60)
    print("  RECALL CEILING EVALUATION")
    print("=" * 60)

    gt = pd.read_csv(os.path.join(RAW_TRAIN_DIR, "train_ground_truth.tsv"), sep="\t")
    cand_file = _candidate_file("train", sample_size)
    cands = pd.read_csv(cand_file, sep="\t")
    cands["candidate_entity_ids"] = cands["candidate_entity_ids"].fillna("")
    cand_map = dict(zip(cands["source1_entity_id"], cands["candidate_entity_ids"]))

    s1 = _load_source("train", "source1", sample_size)
    s2 = _load_source("train", "source2", sample_size)
    s3 = _load_source("train", "source3", sample_size)

    s1_ids = set(s1["entity_id"].values)
    valid_targets = set(s2["entity_id"].values) | set(s3["entity_id"].values)
    del s1, s2, s3
    gc.collect()

    gt_sample = gt[gt["source1_entity_id"].isin(s1_ids)]

    total = 0
    found = 0
    for s1_id, matched in zip(gt_sample["source1_entity_id"], gt_sample["matched_entity_ids"]):
        if pd.isna(matched) or str(matched).strip() == "":
            continue
        true_in_sample = set(str(matched).split(",")) & valid_targets
        if not true_in_sample:
            continue
        our_cands = set(cand_map.get(s1_id, "").split(",")) if cand_map.get(s1_id, "") else set()
        total += len(true_in_sample)
        found += len(true_in_sample & our_cands)

    print(f"Total true matches (within sample): {total}")
    print(f"Found by blocking: {found}")
    print(f"Missed by blocking: {total - found}")
    if total > 0:
        print(f"RECALL CEILING: {found / total * 100:.2f}%")

In [ ]:
MODE = "smoke"   # "smoke" = first 5000 rows of each source; "full" = everything (needs High-RAM)
SAMPLE_SIZE = 5000 if MODE == "smoke" else None   # used by every later step

if MODE == "smoke":
    run_blocking_pipeline(is_test=False, sample_size=5000)
    evaluate_recall(sample_size=5000)
    run_blocking_pipeline(is_test=True, sample_size=5000)   # smoke-size test candidates, so the last steps can be exercised too
elif MODE == "full":
    run_blocking_pipeline(is_test=False, sample_size=None)
    evaluate_recall(sample_size=None)
    run_blocking_pipeline(is_test=True, sample_size=None)
else:
    raise ValueError("MODE must be 'smoke' or 'full'")

## 3. Build training pairs (`build_training_pairs.py`)

Positives = true matches that survived blocking; hard negatives = wrong blocking
candidates; easy negatives = random same-country records.

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
# Every candidate that survives blocking but isn't a true match is, by
# construction, something blocking thought looked plausible -- i.e. already a
# hard negative (similar name/address/tokens/country). We keep all positives
# and cap hard negatives per entity so a handful of entities with huge
# candidate lists don't dominate the training set. Easy/random negatives are
# added on top so the model also sees the "obviously different business"
# case, not just the hard boundary.
HARD_NEG_CAP_PER_ENTITY = 20
EASY_NEG_PER_ENTITY = 2
RANDOM_SEED = 42

In [ ]:
def _candidate_file(prefix, sample_size):
    suffix = "_TEST" if sample_size else ""
    return os.path.join(CAND_DIR, f"{prefix}_candidate_pairs{suffix}.tsv")


def _explode_id_list(df, id_col, list_col, out_col):
    rows = df[df[list_col].notna() & (df[list_col].astype(str).str.strip() != "")].copy()
    if rows.empty:
        return pd.DataFrame(columns=[id_col, out_col])
    rows[list_col] = rows[list_col].astype(str).str.split(",")
    exploded = rows.explode(list_col)[[id_col, list_col]].rename(columns={list_col: out_col})
    return exploded


def build_training_pairs(sample_size=None, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)

    print("Loading candidate pairs and ground truth...")
    cand_file = _candidate_file("train", sample_size)
    cand_df = pd.read_csv(cand_file, sep="\t")
    gt = pd.read_csv(os.path.join(RAW_TRAIN_DIR, "train_ground_truth.tsv"), sep="\t")

    cand_pairs = _explode_id_list(cand_df, "source1_entity_id", "candidate_entity_ids", "candidate_entity_id")
    true_pairs = _explode_id_list(gt, "source1_entity_id", "matched_entity_ids", "candidate_entity_id")

    cand_pairs["key"] = cand_pairs["source1_entity_id"] + "||" + cand_pairs["candidate_entity_id"]
    true_pairs["key"] = true_pairs["source1_entity_id"] + "||" + true_pairs["candidate_entity_id"]
    true_keys = set(true_pairs["key"])

    # Restrict to S1 entities actually present in this candidate file (matters
    # when sample_size subsets the source files for a smoke test).
    s1_ids_in_scope = set(cand_df["source1_entity_id"])
    true_pairs_in_scope = true_pairs[true_pairs["source1_entity_id"].isin(s1_ids_in_scope)]

    missed = true_pairs_in_scope[~true_pairs_in_scope["key"].isin(set(cand_pairs["key"]))]
    if len(missed):
        print(f"WARNING: {len(missed)}/{len(true_pairs_in_scope)} true matches were not produced by "
              f"blocking and cannot be used as positive training examples (this is the same "
              f"gap as blocking's recall-ceiling miss rate).")

    cand_pairs["label"] = cand_pairs["key"].isin(true_keys).astype(np.int8)

    positives = cand_pairs[cand_pairs["label"] == 1][["source1_entity_id", "candidate_entity_id", "label"]]
    hard_neg_pool = cand_pairs[cand_pairs["label"] == 0]

    print(f"Positives (true matches surviving blocking): {len(positives)}")
    print(f"Hard-negative pool (blocking candidates, wrong match): {len(hard_neg_pool)}")

    hard_neg_pool = hard_neg_pool.sample(frac=1.0, random_state=seed)
    hard_negatives = (
        hard_neg_pool.groupby("source1_entity_id", sort=False)
        .head(HARD_NEG_CAP_PER_ENTITY)[["source1_entity_id", "candidate_entity_id", "label"]]
    )
    print(f"Hard negatives kept (<= {HARD_NEG_CAP_PER_ENTITY}/entity): {len(hard_negatives)}")
    del cand_pairs, hard_neg_pool
    gc.collect()

    print("Sampling easy (random) negatives...")
    easy_negatives = _sample_easy_negatives(true_keys, sample_size, EASY_NEG_PER_ENTITY, rng)
    print(f"Easy negatives sampled: {len(easy_negatives)}")

    training_pairs = pd.concat([positives, hard_negatives, easy_negatives], ignore_index=True)
    training_pairs = training_pairs.drop_duplicates(subset=["source1_entity_id", "candidate_entity_id"])
    training_pairs = training_pairs.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    out_file = os.path.join(PAIRS_DIR, f"train_pairs{'_TEST' if sample_size else ''}.tsv")
    training_pairs.to_csv(out_file, sep="\t", index=False)

    print(f"\nSaved {len(training_pairs)} labeled pairs to {out_file}")
    print(f"  positives: {(training_pairs['label']==1).sum()}")
    print(f"  negatives: {(training_pairs['label']==0).sum()} "
          f"(hard: {len(hard_negatives)}, easy: {len(easy_negatives)})")
    return training_pairs

In [ ]:
def _sample_easy_negatives(true_keys, sample_size, k, rng):
    """For each S1 entity, sample k random Source-2/3 records from the SAME
    country that never showed up as a candidate -- the "obviously different
    business" case (e.g. Acme Robotics vs. a random Delhi bakery)."""
    s1 = pd.read_csv(os.path.join(NORM_DIR, "train_source1.tsv"), sep="\t",
                      usecols=["entity_id", "country"], nrows=sample_size)
    s2 = pd.read_csv(os.path.join(NORM_DIR, "train_source2.tsv"), sep="\t",
                      usecols=["entity_id", "country"], nrows=sample_size)
    s3 = pd.read_csv(os.path.join(NORM_DIR, "train_source3.tsv"), sep="\t",
                      usecols=["entity_id", "country"], nrows=sample_size)
    target_pool = pd.concat([s2, s3], ignore_index=True)
    del s2, s3
    gc.collect()

    rows_s1, rows_cand = [], []
    for country, group in s1.groupby("country"):
        pool = target_pool.loc[target_pool["country"] == country, "entity_id"].values
        if len(pool) == 0:
            continue
        n = len(group)
        idx = rng.integers(0, len(pool), size=(n, k))
        sampled = pool[idx]
        s1_ids = group["entity_id"].values
        rows_s1.append(np.repeat(s1_ids, k))
        rows_cand.append(sampled.reshape(-1))

    if not rows_s1:
        return pd.DataFrame(columns=["source1_entity_id", "candidate_entity_id", "label"])

    easy = pd.DataFrame({
        "source1_entity_id": np.concatenate(rows_s1),
        "candidate_entity_id": np.concatenate(rows_cand),
    })
    easy["key"] = easy["source1_entity_id"] + "||" + easy["candidate_entity_id"]

    # Drop any accidental collision with a true match (astronomically rare
    # given pool sizes, but cheap to guarantee); collisions with an existing
    # hard-negative candidate are harmless -- build_training_pairs() dedupes.
    collision = easy["key"].isin(true_keys)
    if collision.any():
        easy = easy[~collision]

    easy["label"] = 0
    return easy[["source1_entity_id", "candidate_entity_id", "label"]]

In [ ]:
# Uses the same MODE as the blocking step above.
if MODE == "smoke":
    build_training_pairs(sample_size=5000)
else:
    build_training_pairs(sample_size=None)

## 4. Feature engineering (`build_features.py`)

For every S1 <-> candidate pair: name / address / country / component / cross-field
features, written as parquet parts to `features/`. Missing information is `NaN`
("unknown"), not 0 -- LightGBM/XGBoost handle it natively.

In [ ]:
!pip install -q -U rapidfuzz

In [ ]:
import os
import re
import gc
import time
import joblib
import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein
from rapidfuzz.process import cpdist
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer


REC_COLS = ["entity_id", "business_address", "country", "norm_name", "norm_name_no_suffix", "norm_address"]
TFIDF_FIT_SAMPLE = 300_000    # IDF is estimated on a sample of records, not every record
CHUNK_SIZE = 1_000_000        # pairs per feature chunk (one parquet part each) -- bounds RAM

# State gazetteers. The SAME state is written differently per source (S1 US: "NC", S3 US:
# "North Carolina"; S1 India: "Madhya Pradesh", S3 India: "MP" or Devanagari), so every alias
# is mapped to one canonical lowercase name -- otherwise state_match is NaN for most S1<->S3
# pairs. Tables are per country because abbreviations collide ("OR" = Oregon vs Odisha).
# Countries not listed (e.g. France) get an empty state -> state_match = NaN ("unknown"),
# never a false mismatch.
_US = """al alabama|ak alaska|az arizona|ar arkansas|ca california|co colorado|ct connecticut|
de delaware|dc district of columbia|fl florida|ga georgia|hi hawaii|id idaho|il illinois|
in indiana|ia iowa|ks kansas|ky kentucky|la louisiana|me maine|md maryland|ma massachusetts|
mi michigan|mn minnesota|ms mississippi|mo missouri|mt montana|ne nebraska|nv nevada|
nh new hampshire|nj new jersey|nm new mexico|ny new york|nc north carolina|nd north dakota|
oh ohio|ok oklahoma|or oregon|pa pennsylvania|ri rhode island|sc south carolina|
sd south dakota|tn tennessee|tx texas|ut utah|vt vermont|va virginia|wa washington|
wv west virginia|wi wisconsin|wy wyoming"""

# canonical name -> "|"-separated aliases (abbreviations + Devanagari/Tamil/Telugu/Kannada/Bengali/... names)
_INDIA = {
    "andhra pradesh": "ap|आंध्र प्रदेश|ఆంధ్రప్రదేశ్", "arunachal pradesh": "ar|अरुणाचल प्रदेश", "assam": "as|असम",
    "bihar": "br|बिहार", "chhattisgarh": "cg|ct|छत्तीसगढ़", "goa": "ga|गोवा", "gujarat": "gj|गुजरात|ગુજરાત",
    "haryana": "hr|hy|हरियाणा", "himachal pradesh": "hp|हिमाचल प्रदेश", "jharkhand": "jh|झारखंड",
    "karnataka": "ka|कर्नाटक|ಕರ್ನಾಟಕ", "kerala": "kl|kr|केरल|കേരളം", "madhya pradesh": "mp|मध्य प्रदेश",
    "maharashtra": "mh|महाराष्ट्र", "manipur": "mn|मणिपुर", "meghalaya": "ml|मेघालय", "mizoram": "mz|मिज़ोरम",
    "nagaland": "nl|नागालैंड", "odisha": "od|or|orissa|ओडिशा|उड़ीसा|ଓଡ଼ିଶା", "punjab": "pb|पंजाब|ਪੰਜਾਬ",
    "rajasthan": "rj|राजस्थान", "sikkim": "sk|सिक्किम", "tamil nadu": "tn|तमिलनाडु|तमिल नाडु|தமிழ்நாடு",
    "telangana": "ts|tg|तेलंगाना|తెలంగాణ", "tripura": "tr|त्रिपुरा", "uttar pradesh": "up|उत्तर प्रदेश",
    "uttarakhand": "uk|ut|उत्तराखंड", "west bengal": "wb|पश्चिम बंगाल|পশ্চিমবঙ্গ", "delhi": "dl|दिल्ली",
    "jammu and kashmir": "jk|जम्मू और कश्मीर", "ladakh": "ld|लद्दाख", "chandigarh": "ch|चंडीगढ़",
    "puducherry": "py|pondicherry|पुडुचेरी", "andaman and nicobar islands": "an",
    "dadra and nagar haveli": "dn", "daman and diu": "dd", "lakshadweep": "lk",
}

STATE_TABLES = {"US": {}, "India": {}}
for _entry in _US.replace("\n", "").split("|"):
    _abbr, _name = _entry.split(" ", 1)
    STATE_TABLES["US"][_abbr] = STATE_TABLES["US"][_name] = _name
for _name, _aliases in _INDIA.items():
    for _alias in [_name] + _aliases.split("|"):
        STATE_TABLES["India"][_alias] = _name

# fallback (address has no comma structure): scan the normalized text for full English names
_STATE_RES = {
    c: re.compile(r"\b(" + "|".join(sorted((k for k in t if len(k) > 2 and k.isascii()), key=len, reverse=True)) + r")\b")
    for c, t in STATE_TABLES.items()
}
_PUNCT = re.compile(r"[!-/:-@\[-`{-~।]")   # ASCII punctuation + danda; keeps Devanagari vowel signs intact

In [ ]:
# PER-RECORD ATTRIBUTES (computed once per unique entity, not per pair)
def _clean_part(p):
    return re.sub(r"\s+", " ", _PUNCT.sub(" ", p.lower().replace("&", " and "))).strip()


def _state_of_part(part, table):
    """part = one cleaned comma-separated chunk of the raw address -> canonical state or ""."""
    letters = " ".join(w for w in part.split() if not w.isdigit())   # "nc 27260" -> "nc"
    return table.get(letters, "")


def extract_geo(raw, norm, country):
    """-> (state, city, postal, house). "" means 'not found'. Heuristic, order-robust:
    the state is the rightmost comma-chunk that IS a state; the city is the chunk next
    to it (before it, else after it) that has no digits."""
    table = STATE_TABLES.get(country, {})
    parts = [_clean_part(p) for p in raw.split(",")]

    state, si = "", -1
    if table:
        for i in range(len(parts) - 1, -1, -1):
            state = _state_of_part(parts[i], table)
            if state:
                si = i
                break
        if not state:                                   # no comma structure: scan the normalized text
            m = _STATE_RES[country].findall(norm)
            if m:
                state = table[m[-1]]
            else:
                toks = norm.split()
                state = next((table[t] for t in toks[-1:] + toks[:1] if len(t) == 2 and t in table), "")

    city = ""
    if si >= 0:
        for j in (si - 1, si + 1):
            if 0 <= j < len(parts) and parts[j] and not any(ch.isdigit() for ch in parts[j]):
                city = parts[j]
                break

    nums = re.findall(r"\b\d+\b", norm)
    # PIN (India, 6 digits) / postal code (5 digits, only if a house number precedes it --
    # a lone leading 5-digit number is a US house number like "17560 Ellis Road").
    postal = next((n for n in nums if len(n) == 6), "") or next((n for n in nums[1:] if len(n) == 5), "")
    house = next((n for n in nums if n != postal and len(n) <= 5), "")
    return state, city, postal, house


def add_record_attributes(r):
    r["name"] = r["norm_name_no_suffix"].where(r["norm_name_no_suffix"] != "", r["norm_name"])
    r["name_first_tok"] = r["name"].str.split(n=1).str[0].fillna("")
    r["name_nonlatin"] = r["name"].str.contains(r"[^\x00-ɏ]", regex=True).astype(np.float32)

    geo = [extract_geo(a, b, c) for a, b, c in zip(r["business_address"].values, r["norm_address"].values, r["country"].values)]
    geo = pd.DataFrame(geo, columns=["state", "city", "postal", "house"], index=r.index)
    return r.join(geo)

In [ ]:
# PAIR-LEVEL PRIMITIVES
def _obj(s):
    return s.to_numpy(dtype=object)


def _eq(x, y):
    """1.0 / 0.0 where both sides are non-empty, NaN ("unknown") otherwise."""
    out = (x == y).astype(np.float32)
    out[(x == "") | (y == "")] = np.nan
    return out


def _sim(scorer, x, y, scale=1.0):
    """Aligned pairwise similarity x[i] vs y[i], multithreaded in C++. NaN if either side empty."""
    out = cpdist(x.tolist(), y.tolist(), scorer=scorer, dtype=np.float32, workers=-1) * np.float32(scale)
    out[(x == "") | (y == "")] = np.nan
    return out


def _rowdot(X, a, b):
    return np.asarray(X[a].multiply(X[b]).sum(axis=1)).ravel().astype(np.float32)


def _cosine(X, a, b, x, y):
    out = _rowdot(X, a, b)
    out[(x == "") | (y == "")] = np.nan
    return out


def _jaccard(H, a, b):
    inter = _rowdot(H, a, b)
    size = np.asarray(H.sum(axis=1)).ravel().astype(np.float32)
    union = size[a] + size[b] - inter
    out = np.full(len(a), np.nan, dtype=np.float32)
    ok = union > 0
    out[ok] = inter[ok] / union[ok]
    return out


def _abs_len_diff(x, y):
    lx = np.fromiter((len(s) for s in x), dtype=np.float32, count=len(x))
    ly = np.fromiter((len(s) for s in y), dtype=np.float32, count=len(y))
    return np.abs(lx - ly)

In [ ]:
# VECTORIZERS
def fit_vectorizers(rec, seed=42):
    s = rec.sample(min(TFIDF_FIT_SAMPLE, len(rec)), random_state=seed)
    names = s["norm_name_no_suffix"].where(s["norm_name_no_suffix"] != "", s["norm_name"])
    print(f"Fitting TF-IDF vocabularies on {len(s)} records...")
    return {
        # names: char n-grams survive typos / word-order changes
        "name_tfidf": TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), min_df=2, sublinear_tf=True,
                                      dtype=np.float32).fit(names),
        # addresses: word-level, so IDF down-weights "street", "road", "nagar", ...
        "addr_tfidf": TfidfVectorizer(analyzer="word", token_pattern=r"\S+", min_df=2, sublinear_tf=True,
                                      dtype=np.float32).fit(s["norm_address"]),
        # Jaccard needs no vocabulary: hash tokens -> binary sets (no OOV tokens dropped)
        "tok": HashingVectorizer(token_pattern=r"\S+", binary=True, norm=None, alternate_sign=False,
                                 n_features=2 ** 20, dtype=np.float32),
        "num": HashingVectorizer(token_pattern=r"\b\d+\b", binary=True, norm=None, alternate_sign=False,
                                 n_features=2 ** 20, dtype=np.float32),
    }


def get_vectorizers(rec, sample_size, refit=False):
    """Fit once (on the train pair records) and reuse for validation + test, so a feature means
    the same thing everywhere. Pickled, so a runtime restart keeps them."""
    path = os.path.join(FEATURES_DIR, f"vectorizers{'_TEST' if sample_size else ''}.joblib")
    if os.path.exists(path) and not refit:
        return joblib.load(path)
    vecs = fit_vectorizers(rec)
    joblib.dump(vecs, path)
    return vecs

In [ ]:
# FEATURES FOR A CHUNK OF PAIRS
def compute_features(pairs, rec, vecs):
    ids = pd.unique(np.concatenate([pairs["source1_entity_id"].values, pairs["candidate_entity_id"].values]))
    r = add_record_attributes(rec.loc[ids].copy())
    pos = pd.Series(np.arange(len(r)), index=r.index)
    a = pos.loc[pairs["source1_entity_id"]].to_numpy()
    b = pos.loc[pairs["candidate_entity_id"]].to_numpy()

    name, addr = _obj(r["name"]), _obj(r["norm_address"])
    na, nb = name[a], name[b]
    aa, ab = addr[a], addr[b]

    Xn = vecs["name_tfidf"].transform(r["name"])
    Xa = vecs["addr_tfidf"].transform(r["norm_address"])
    Hn = vecs["tok"].transform(r["name"])
    Ha = vecs["tok"].transform(r["norm_address"])
    Hnum = vecs["num"].transform(r["norm_address"])

    f = {
        "source1_entity_id": pairs["source1_entity_id"].values,
        "candidate_entity_id": pairs["candidate_entity_id"].values,
        "cand_is_s3": pairs["candidate_entity_id"].str.startswith("S3-").to_numpy().astype(np.float32),
    }

    # ---- country (open-set: plain string equality, no hard-coded country list) ----
    country = _obj(r["country"])
    f["country_match"] = (country[a] == country[b]).astype(np.float32)

    # ---- name ----
    f["name_exact"] = _eq(na, nb)
    f["name_exact_with_suffix"] = _eq(_obj(r["norm_name"])[a], _obj(r["norm_name"])[b])
    f["name_lev"] = _sim(Levenshtein.normalized_similarity, na, nb)
    f["name_jaro_winkler"] = _sim(JaroWinkler.similarity, na, nb)
    f["name_jaccard"] = _jaccard(Hn, a, b)
    f["name_token_set"] = _sim(fuzz.token_set_ratio, na, nb, 0.01)     # "abc tech india pvt" ~ "abc tech pvt india"
    f["name_token_sort"] = _sim(fuzz.token_sort_ratio, na, nb, 0.01)
    f["name_tfidf"] = _cosine(Xn, a, b, na, nb)
    f["name_first_token_match"] = _eq(_obj(r["name_first_tok"])[a], _obj(r["name_first_tok"])[b])
    f["name_len_diff"] = _abs_len_diff(na, nb)
    # If exactly one side is non-Latin script (e.g. Devanagari vs English), the name features are
    # unreliable -> lets the model lean on address instead.
    f["name_script_mismatch"] = np.abs(r["name_nonlatin"].to_numpy()[a] - r["name_nonlatin"].to_numpy()[b])

    # ---- address ----
    f["address_exact"] = _eq(aa, ab)
    f["address_lev"] = _sim(Levenshtein.normalized_similarity, aa, ab)
    f["address_jaro_winkler"] = _sim(JaroWinkler.similarity, aa, ab)
    f["address_jaccard"] = _jaccard(Ha, a, b)
    f["address_token_set"] = _sim(fuzz.token_set_ratio, aa, ab, 0.01)  # robust to component reordering
    f["address_tfidf"] = _cosine(Xa, a, b, aa, ab)
    f["address_num_jaccard"] = _jaccard(Hnum, a, b)
    f["address_len_diff"] = _abs_len_diff(aa, ab)

    # ---- address components (NaN = missing on at least one side, i.e. unknown, not a mismatch) ----
    f["house_number_match"] = _eq(_obj(r["house"])[a], _obj(r["house"])[b])
    f["pin_match"] = _eq(_obj(r["postal"])[a], _obj(r["postal"])[b])
    f["state_match"] = _eq(_obj(r["state"])[a], _obj(r["state"])[b])
    city = _obj(r["city"])
    f["city_match"] = _eq(city[a], city[b])
    f["city_jaro_winkler"] = _sim(JaroWinkler.similarity, city[a], city[b])

    # ---- cross-field ----
    name_score = pd.DataFrame({k: f[k] for k in ("name_jaro_winkler", "name_tfidf", "name_token_set")}).mean(axis=1).to_numpy(np.float32)
    addr_score = pd.DataFrame({k: f[k] for k in ("address_jaro_winkler", "address_tfidf", "address_token_set")}).mean(axis=1).to_numpy(np.float32)
    f["name_score"], f["address_score"] = name_score, addr_score
    f["name_x_address"] = name_score * addr_score
    f["name_plus_address"] = name_score + addr_score
    f["name_plus_address_plus_country"] = name_score + addr_score + f["country_match"]
    f["both_strong"] = ((name_score > 0.8) & (addr_score > 0.8)).astype(np.float32)
    f["name_strong_addr_weak"] = ((name_score > 0.8) & (addr_score < 0.5)).astype(np.float32)
    f["name_weak_addr_strong"] = ((name_score < 0.5) & (addr_score > 0.8)).astype(np.float32)

    out = pd.DataFrame(f)
    if "label" in pairs.columns:
        out["label"] = pairs["label"].to_numpy(np.int8)
    return out

In [ ]:
# I/O + DRIVER
def load_pairs(prefix, sample_size):
    suffix = "_TEST" if sample_size else ""
    if prefix == "train":
        pairs = pd.read_csv(os.path.join(PAIRS_DIR, f"train_pairs{suffix}.tsv"), sep="\t", dtype=str)
        pairs["label"] = pairs["label"].astype(np.int8)
        return pairs
    # test: no labels -- explode the blocking candidate lists
    cand = pd.read_csv(os.path.join(CAND_DIR, f"test_candidate_pairs{suffix}.tsv"), sep="\t", dtype=str)
    cand = cand[cand["candidate_entity_ids"].fillna("") != ""]
    cand = cand.assign(candidate_entity_id=cand["candidate_entity_ids"].str.split(",")).explode("candidate_entity_id")
    return cand[["source1_entity_id", "candidate_entity_id"]].reset_index(drop=True)


def load_records(prefix, sample_size, needed_ids):
    frames = []
    for src in ("source1", "source2", "source3"):
        path = os.path.join(NORM_DIR, f"{prefix}_{src}.tsv")
        # keep_default_na=False: empty cells stay "" (never NaN), so every string op below is safe
        kw = dict(sep="\t", usecols=REC_COLS, dtype=str, keep_default_na=False)
        if sample_size:
            df = pd.read_csv(path, nrows=sample_size, **kw)
            frames.append(df[df["entity_id"].isin(needed_ids)])
        else:
            for chunk in pd.read_csv(path, chunksize=1_000_000, **kw):
                frames.append(chunk[chunk["entity_id"].isin(needed_ids)])
    rec = pd.concat(frames, ignore_index=True).drop_duplicates("entity_id").set_index("entity_id")
    print(f"Loaded {len(rec)} records")
    return rec


def build_features(prefix="train", sample_size=None, chunk_size=CHUNK_SIZE, pairs=None, save=True,
                   refit_vectorizers=False):
    """save=True : write parquet parts to features/<tag>/ and return that directory.
    save=False: return one in-memory DataFrame (used for the validation sets).
    pairs=None : load the default pair table for `prefix` (train_pairs / exploded test candidates)."""
    t0 = time.time()
    tag = f"{prefix}{'_TEST' if sample_size else ''}"
    print(f"\n{'=' * 60}\n  FEATURE ENGINEERING — {tag}\n{'=' * 60}")

    if pairs is None:
        pairs = load_pairs(prefix, sample_size)
    print(f"Pairs: {len(pairs)}")
    needed = set(pairs["source1_entity_id"]) | set(pairs["candidate_entity_id"])
    rec = load_records(prefix, sample_size, needed)
    del needed
    gc.collect()

    known = pairs["source1_entity_id"].isin(rec.index) & pairs["candidate_entity_id"].isin(rec.index)
    if not known.all():
        print(f"WARNING: dropping {(~known).sum()} pairs whose records were not found")
        pairs = pairs[known].reset_index(drop=True)

    vecs = get_vectorizers(rec, sample_size, refit=refit_vectorizers)

    if not save:
        frames = [compute_features(pairs.iloc[s:s + chunk_size], rec, vecs) for s in range(0, len(pairs), chunk_size)]
        return pd.concat(frames, ignore_index=True)

    out_dir = os.path.join(FEATURES_DIR, tag)
    os.makedirs(out_dir, exist_ok=True)
    for old in os.listdir(out_dir):               # this script's own previous parts
        if old.startswith("part-"):
            os.remove(os.path.join(out_dir, old))

    n_parts = 0
    for start in range(0, len(pairs), chunk_size):
        feats = compute_features(pairs.iloc[start:start + chunk_size], rec, vecs)
        feats.to_parquet(os.path.join(out_dir, f"part-{n_parts:04d}.parquet"), index=False)
        n_parts += 1
        print(f"  chunk {n_parts}: {start + len(feats)}/{len(pairs)} pairs ({time.time() - t0:.0f}s)")
        del feats
        gc.collect()

    print(f"\nSaved {n_parts} part(s) to {out_dir}  (read back with pd.read_parquet('{out_dir}'))")
    return out_dir

In [ ]:
build_features("train", sample_size=SAMPLE_SIZE, refit_vectorizers=True)

TRAIN_TAG = "train_TEST" if SAMPLE_SIZE else "train"
head = pd.read_parquet(os.path.join(FEATURES_DIR, TRAIN_TAG, "part-0000.parquet"))   # first part only: enough for a sanity check
print(head.shape)
head.drop(columns=["source1_entity_id", "candidate_entity_id"]).groupby("label").mean().T.round(3)

## 5. Validation set (test-like)

The training pairs are *down-sampled* (<= 20 hard negatives per entity + random easy negatives), so they
don't look like test time, where the model sees **every** blocking candidate. So we hold out a slice of
S1 entities and score **all** of their candidates. Half is used for early stopping, half for picking the
decision threshold. The split is by S1 entity, so nothing leaks between train and validation.

In [ ]:
import glob
import shutil
import subprocess
import tempfile
import matplotlib.pyplot as plt
import lightgbm as lgb
import pyarrow.parquet as pq

VAL_FRACTION = 0.10
MAX_VAL_ENTITIES = 50_000     # cap: features are computed for ALL candidates of these entities
RANDOM_SEED = 42
ID_COLS = ["source1_entity_id", "candidate_entity_id"]


def load_ground_truth(sample_size):
    """True matches as (source1_entity_id, candidate_entity_id) rows. In smoke mode only matches whose
    S2/S3 record is inside the sampled rows can be found, so the rest are dropped (same as evaluate_recall)."""
    gt = pd.read_csv(os.path.join(RAW_TRAIN_DIR, "train_ground_truth.tsv"), sep="\t", dtype=str)
    gt = gt[gt["matched_entity_ids"].fillna("") != ""]
    gt = gt.assign(candidate_entity_id=gt["matched_entity_ids"].str.split(",")).explode("candidate_entity_id")
    gt = gt[ID_COLS]
    if sample_size:
        targets = set()
        for src in ("source2", "source3"):
            ids = pd.read_csv(os.path.join(NORM_DIR, f"train_{src}.tsv"), sep="\t", usecols=["entity_id"],
                              nrows=sample_size, dtype=str)["entity_id"]
            targets |= set(ids)
        gt = gt[gt["candidate_entity_id"].isin(targets)]
    return gt.reset_index(drop=True)


def split_s1_entities(sample_size, seed=RANDOM_SEED):
    """-> (early-stopping S1 ids, threshold-tuning S1 ids)"""
    ids = pd.read_csv(os.path.join(NORM_DIR, "train_source1.tsv"), sep="\t", usecols=["entity_id"],
                      nrows=sample_size, dtype=str)["entity_id"].to_numpy()
    np.random.default_rng(seed).shuffle(ids)
    n_val = min(int(len(ids) * VAL_FRACTION), MAX_VAL_ENTITIES)
    return ids[: n_val // 2], ids[n_val // 2: n_val]


def build_validation_features(s1_ids, gt, sample_size):
    """Features + labels for ALL blocking candidates of `s1_ids` (no capping / negative sampling),
    plus the true-match count per S1 entity (including matches blocking missed)."""
    s1_ids = list(s1_ids)
    cand = pd.read_csv(_candidate_file("train", sample_size), sep="\t", dtype=str)
    cand = cand[cand["source1_entity_id"].isin(set(s1_ids)) & (cand["candidate_entity_ids"].fillna("") != "")]
    pairs = cand.assign(candidate_entity_id=cand["candidate_entity_ids"].str.split(",")).explode("candidate_entity_id")
    pairs = pairs[ID_COLS].reset_index(drop=True)

    true_keys = set(gt["source1_entity_id"] + "||" + gt["candidate_entity_id"])
    pairs["label"] = (pairs["source1_entity_id"] + "||" + pairs["candidate_entity_id"]).isin(true_keys).astype(np.int8)

    feats = build_features("train", sample_size, pairs=pairs, save=False)
    ntrue = gt[gt["source1_entity_id"].isin(set(s1_ids))].groupby("source1_entity_id").size().reindex(s1_ids, fill_value=0)
    return feats, ntrue


gt = load_ground_truth(SAMPLE_SIZE)
es_ids, thr_ids = split_s1_entities(SAMPLE_SIZE)
val_ids = set(es_ids) | set(thr_ids)
print(f"Held-out S1 entities: {len(es_ids)} (early stopping) + {len(thr_ids)} (threshold)")

val_es, ntrue_es = build_validation_features(es_ids, gt, SAMPLE_SIZE)
val_thr, ntrue_thr = build_validation_features(thr_ids, gt, SAMPLE_SIZE)
print(f"Validation pairs: {len(val_es)} + {len(val_thr)} | positives: {int(val_es.label.sum())} + {int(val_thr.label.sum())}")
print(f"Blocking recall ceiling on the threshold set: {val_thr.label.sum() / max(ntrue_thr.sum(), 1):.3f} "
      f"| singletons: {(ntrue_thr == 0).mean():.3f}")

## 6. Train the matcher (LightGBM, MIT licence)

Trained on the labelled pairs from the non-held-out S1 entities. All positives are kept; if the table is
larger than `MAX_TRAIN_ROWS`, negatives are randomly down-sampled. `NaN` (unknown) is handled natively.

In [ ]:
MAX_TRAIN_ROWS = 6_000_000


def load_training_table(sample_size, exclude_ids, seed=RANDOM_SEED):
    tag = "train_TEST" if sample_size else "train"
    parts = sorted(glob.glob(os.path.join(FEATURES_DIR, tag, "part-*.parquet")))
    total = sum(pq.ParquetFile(p).metadata.num_rows for p in parts)
    neg_frac = min(1.0, MAX_TRAIN_ROWS / total)
    rng = np.random.default_rng(seed)
    frames = []
    for p in parts:                                   # part by part: never holds the id strings of all rows at once
        df = pd.read_parquet(p)
        df = df[~df["source1_entity_id"].isin(exclude_ids)]
        keep = (df["label"] == 1).to_numpy() | (rng.random(len(df)) < neg_frac)
        frames.append(df.loc[keep].drop(columns=ID_COLS))
    return pd.concat(frames, ignore_index=True)


train = load_training_table(SAMPLE_SIZE, val_ids)
FEATURE_COLS = [c for c in train.columns if c != "label"]
print(f"Training rows: {len(train)} | positives: {int(train.label.sum())} | features: {len(FEATURE_COLS)}")

params = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05, num_leaves=63, min_child_samples=50,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, reg_lambda=1.0, seed=RANDOM_SEED, verbose=-1,
)
dtrain = lgb.Dataset(train[FEATURE_COLS], train["label"])
dval = lgb.Dataset(val_es[FEATURE_COLS], val_es["label"], reference=dtrain)
model = lgb.train(params, dtrain, num_boost_round=2000, valid_sets=[dval],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)])
print("best iteration:", model.best_iteration)
MODELS = {"lightgbm": model}

pd.Series(model.feature_importance("gain"), index=FEATURE_COLS).sort_values(ascending=False).head(15).round(0)

### 6b. Alternatives to compare (XGBoost, logistic regression, ExtraTrees, blend)

All are permissively licensed (LightGBM MIT, XGBoost Apache-2.0, scikit-learn BSD) and tiny compared with the 8B-parameter
limit. The simple models are trained on a sub-sample (`BASELINE_MAX_ROWS`) because they don't scale to millions of rows;
they exist to show whether the boosted trees are really needed. Section 7 scores every model on the same held-out entities.

In [ ]:
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score

BASELINE_MAX_ROWS = 500_000


def subsample(df, n, seed=RANDOM_SEED):
    """Keep every positive, down-sample negatives to at most n rows in total."""
    if len(df) <= n:
        return df
    pos = df[df["label"] == 1]
    neg = df[df["label"] == 0].sample(max(n - len(pos), 0), random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1.0, random_state=seed)


# --- XGBoost (native NaN handling, early stopping on the same early-stopping entities as LightGBM)
MODELS["xgboost"] = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=8, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, tree_method="hist", n_jobs=-1, eval_metric="logloss", early_stopping_rounds=50,
    random_state=RANDOM_SEED,
).fit(train[FEATURE_COLS], train["label"], eval_set=[(val_es[FEATURE_COLS], val_es["label"])], verbose=100)

# --- baselines on a sub-sample. NaN ("unknown") is imputed; the missing-indicators keep that information for the linear model.
small = subsample(train, BASELINE_MAX_ROWS)
MODELS["logreg"] = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), StandardScaler(),
    LogisticRegression(max_iter=500),
).fit(small[FEATURE_COLS], small["label"])
MODELS["extra_trees"] = make_pipeline(
    SimpleImputer(strategy="constant", fill_value=-1, keep_empty_features=True),
    ExtraTreesClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_SEED),
).fit(small[FEATURE_COLS], small["label"])
del small


def predict_proba(name, X):
    """P(same business) for the rows of X (a features DataFrame) under model `name`."""
    X = X[FEATURE_COLS]
    if name == "lightgbm":
        return MODELS[name].predict(X, num_iteration=MODELS[name].best_iteration)
    if name == "blend_lgbm_xgb":
        return 0.5 * (predict_proba("lightgbm", X) + predict_proba("xgboost", X))
    return MODELS[name].predict_proba(X)[:, 1]


print("models ready:", list(MODELS) + ["blend_lgbm_xgb"])

## 7. Compare the models and pick the decision threshold (macro-F0.5)

`threshold = 0.5` is **not** automatically right for F0.5, which rewards precision (a false merge costs more than a
missed match). So every model is swept over thresholds 0.05 ... 0.99 and scored with the challenge metric on the held-out
threshold set; the best (model, threshold) wins.

Per S1 entity: `F0.5 = 1.25*TP / (0.25*n_true + n_pred)`. An entity with no true matches scores 1.0 if we predict
nothing and 0.0 if we predict anything. The score is the mean over **all** entities in the set (singletons included),
computed against the *full* ground truth, so matches that blocking missed still count against us.

In [ ]:
def macro_f05(scored, ntrue, thr):
    """scored: DataFrame(source1_entity_id, prob, label) for every candidate; ntrue: Series indexed by ALL eval S1 ids."""
    codes, uniques = pd.factorize(scored["source1_entity_id"])
    pred = (scored["prob"].to_numpy() >= thr)
    n_pred = np.zeros(len(ntrue))
    tp = np.zeros(len(ntrue))
    pos = ntrue.index.get_indexer(uniques)
    n_pred[pos] = np.bincount(codes, weights=pred, minlength=len(uniques))
    tp[pos] = np.bincount(codes, weights=pred & (scored["label"].to_numpy() == 1), minlength=len(uniques))
    denom = 0.25 * ntrue.to_numpy() + n_pred
    f = np.where(denom > 0, 1.25 * tp / np.maximum(denom, 1e-9), 1.0)   # denom == 0: no truth, no prediction -> 1.0
    return float(f.mean())


grid = np.round(np.arange(0.05, 1.0, 0.01), 2)
COARSE = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
MODEL_NAMES = list(MODELS) + ["blend_lgbm_xgb"]
y_thr = val_thr["label"].to_numpy()

sweeps, rows = {}, []
for name in MODEL_NAMES:
    prob = predict_proba(name, val_thr)
    scored = pd.DataFrame({"source1_entity_id": val_thr["source1_entity_id"].to_numpy(), "label": y_thr, "prob": prob})
    sweeps[name] = np.array([macro_f05(scored, ntrue_thr, t) for t in grid])
    rows.append({
        "model": name,
        "best_thr": grid[sweeps[name].argmax()],
        "macro_F0.5": sweeps[name].max(),
        "F0.5@0.5": macro_f05(scored, ntrue_thr, 0.5),
        "avg_precision": average_precision_score(y_thr, prob) if y_thr.any() else np.nan,
    })
comparison = pd.DataFrame(rows).set_index("model").sort_values("macro_F0.5", ascending=False, kind="stable")

print(f"Predict-nothing baseline (= singleton fraction): {(ntrue_thr == 0).mean():.4f}\n")
print(comparison.round(4), "\n")
print("macro-F0.5 at fixed thresholds:")
print(pd.DataFrame({n: [sweeps[n][int(np.abs(grid - t).argmin())] for t in COARSE] for n in MODEL_NAMES}, index=COARSE).round(4))

BEST_MODEL = comparison.index[0]
BEST_THR = float(comparison.loc[BEST_MODEL, "best_thr"])
BEST_SCORE = float(comparison.loc[BEST_MODEL, "macro_F0.5"])
print(f"\nWINNER: {BEST_MODEL} @ threshold {BEST_THR:.2f} -> macro-F0.5 = {BEST_SCORE:.4f}")

plt.figure(figsize=(7, 3.5))
for n in MODEL_NAMES:
    plt.plot(grid, sweeps[n], label=n, lw=2.5 if n == BEST_MODEL else 1)
plt.axvline(BEST_THR, color="r", ls="--")
plt.xlabel("threshold")
plt.ylabel("macro F0.5")
plt.title("Threshold sweep (held-out entities)")
plt.legend(fontsize=7)
plt.show()

val_thr["prob"] = predict_proba(BEST_MODEL, val_thr)

## 7b. Exclusive assignment (each S2/S3 record belongs to at most one S1 entity)

In the training ground truth every S2/S3 record is matched by **exactly one** S1 entity (checked on 300k S1 rows: max 1
S1 per target). So if two S1 entities both claim the same S2/S3 record, at most one claim is right: keep only the claimant
with the highest probability. This removes false merges (precision) without touching real matches.

The validation set only contains a slice of the S1 entities, so to test this fairly we also score the *competing* S1
entities' pairs for every target the slice claims. It is adopted only if it does not lower the validation macro-F0.5.

In [ ]:
MIN_EXCL_THR = 0.30     # competitors are only fetched/scored for targets claimed at >= this probability


def pair_key(d):
    return d["source1_entity_id"] + "||" + d["candidate_entity_id"]


def reverse_claimants(targets, sample_size, chunksize=200_000):
    """Every blocking pair (S1, target) whose target is in `targets` -- i.e. all S1 entities that could claim them."""
    targets, out = set(targets), []
    for ch in pd.read_csv(_candidate_file("train", sample_size), sep="\t", dtype=str, chunksize=chunksize):
        ch = ch[ch["candidate_entity_ids"].fillna("") != ""]
        ex = ch.assign(candidate_entity_id=ch["candidate_entity_ids"].str.split(",")).explode("candidate_entity_id")
        out.append(ex.loc[ex["candidate_entity_id"].isin(targets), ID_COLS])
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=ID_COLS)


def keep_best_claimant(claims, thr):
    """claims: DataFrame(source1_entity_id, candidate_entity_id, prob). Among claims with prob >= thr keep, per target,
    only the highest-probability S1 entity. Returns the surviving claims."""
    c = claims[claims["prob"] >= thr].sort_values("prob", ascending=False, kind="mergesort")
    return c.drop_duplicates("candidate_entity_id")


def with_exclusive_probs(scored, claims_sorted, thr):
    """Copy of `scored` whose prob is zeroed for pairs that lose their target to a higher-probability S1 entity."""
    top = claims_sorted[claims_sorted["prob"] >= thr].drop_duplicates("candidate_entity_id")
    winners = set(pair_key(top))
    return scored.assign(prob=np.where(pair_key(scored).isin(winners), scored["prob"], 0.0))


# competitors for every target the held-out slice claims
targets = set(val_thr.loc[val_thr["prob"] >= MIN_EXCL_THR, "candidate_entity_id"])
rev = reverse_claimants(targets, SAMPLE_SIZE)
rev = rev[~pair_key(rev).isin(set(pair_key(val_thr)))].reset_index(drop=True)      # pairs not scored yet
print(f"{len(targets)} contested targets -> {len(rev)} competing pairs to score")
if len(rev):
    rev_feats = build_features("train", SAMPLE_SIZE, pairs=rev, save=False)
    rev_feats["prob"] = predict_proba(BEST_MODEL, rev_feats)
    rev_claims = rev_feats[ID_COLS + ["prob"]]
else:
    rev_claims = pd.DataFrame(columns=ID_COLS + ["prob"])
claims = pd.concat([val_thr[ID_COLS + ["prob"]], rev_claims], ignore_index=True)
claims = claims.sort_values("prob", ascending=False, kind="mergesort")

grid_ex = grid[grid >= MIN_EXCL_THR]
scored_thr = val_thr[["source1_entity_id", "candidate_entity_id", "label", "prob"]]
scores_ex = np.array([macro_f05(with_exclusive_probs(scored_thr, claims, t), ntrue_thr, t) for t in grid_ex])
best_ex_thr, best_ex = float(grid_ex[scores_ex.argmax()]), float(scores_ex.max())

# ">=": the data guarantees one owner per target, so on a tie the exclusive rule is the safer choice
USE_EXCLUSIVE = best_ex >= BEST_SCORE
FINAL_THR = best_ex_thr if USE_EXCLUSIVE else BEST_THR
FINAL_SCORE = max(best_ex, BEST_SCORE) if USE_EXCLUSIVE else BEST_SCORE
print(f"plain     : threshold {BEST_THR:.2f} -> macro-F0.5 {BEST_SCORE:.4f}")
print(f"exclusive : threshold {best_ex_thr:.2f} -> macro-F0.5 {best_ex:.4f}")
print(f"USE_EXCLUSIVE = {USE_EXCLUSIVE} | FINAL: {BEST_MODEL} @ {FINAL_THR:.2f} (validation macro-F0.5 {FINAL_SCORE:.4f})")

val_final = with_exclusive_probs(val_thr, claims, FINAL_THR) if USE_EXCLUSIVE else val_thr

joblib.dump({"models": MODELS, "best_model": BEST_MODEL, "threshold": FINAL_THR, "exclusive": USE_EXCLUSIVE,
             "feature_cols": FEATURE_COLS}, os.path.join(MODEL_DIR, "matcher.joblib"))
pd.Series({"model": BEST_MODEL, "threshold": FINAL_THR, "exclusive": USE_EXCLUSIVE, "val_macro_f05": FINAL_SCORE}
          ).to_json(os.path.join(MODEL_DIR, "threshold.json"))

## 7c. Error analysis

Where do we lose points: blocking (never a candidate), the model (missed match), or false merges? Plus the worst
false positives / missed matches, for the write-up.

In [ ]:
va = val_final[ID_COLS + ["label", "prob", "cand_is_s3"]].copy()
va["pred"] = va["prob"] >= FINAL_THR

n_true, reachable = int(ntrue_thr.sum()), int(va["label"].sum())
tp = int((va["pred"] & (va["label"] == 1)).sum())
fp = int((va["pred"] & (va["label"] == 0)).sum())
print(f"true matches                : {n_true}")
print(f"  lost to blocking          : {n_true - reachable}  (never a candidate)")
print(f"  lost to the model         : {reachable - tp}  (candidate, but prob < {FINAL_THR:.2f})")
print(f"  found (TP)                : {tp}  -> recall {tp / max(n_true, 1):.3f}")
print(f"false positives (bad merges): {fp}  -> precision {tp / max(tp + fp, 1):.3f}")

va["source"] = np.where(va["cand_is_s3"] == 1, "S3", "S2")
va["is_tp"] = va["pred"] & (va["label"] == 1)
va["is_fp"] = va["pred"] & (va["label"] == 0)
by_src = va.groupby("source").agg(pairs=("label", "size"), true=("label", "sum"), TP=("is_tp", "sum"), FP=("is_fp", "sum"))
print("\nby candidate source:\n", by_src)

fp_rows = va[va["pred"] & (va["label"] == 0)].nlargest(8, "prob").assign(kind="FALSE POSITIVE")
fn_rows = va[~va["pred"] & (va["label"] == 1)].nsmallest(8, "prob").assign(kind="MISSED MATCH")
ex = pd.concat([fp_rows, fn_rows])
if len(ex):
    rec = load_records("train", SAMPLE_SIZE, set(ex["source1_entity_id"]) | set(ex["candidate_entity_id"]))
    show = lambda i: f"{rec.at[i, 'norm_name']} | {rec.at[i, 'business_address']} | {rec.at[i, 'country']}"
    for _, r in ex.iterrows():
        print(f"\n[{r['kind']}] p={r['prob']:.3f}\n  S1  : {show(r['source1_entity_id'])}\n  cand: {show(r['candidate_entity_id'])}")

## 8. Predict the test set and write the submission files

* `output/candidate_pairs.tsv` = the exact blocking candidate set the model scores (copied from `candidates/`).
* `output/matching_results.tsv` = candidates with probability >= the tuned threshold (and, if 7b adopted it, only the
  highest-probability claimant of each S2/S3 record); **one row per S1 test entity** (empty list when nothing passes).

In [ ]:
def write_submission(model_name, thr, sample_size, exclusive):
    tag = "test_TEST" if sample_size else "test"
    parts = sorted(glob.glob(os.path.join(FEATURES_DIR, tag, "part-*.parquet")))
    kept = []
    for p in parts:
        f = pd.read_parquet(p)
        prob = predict_proba(model_name, f)
        m = prob >= thr
        kept.append(f.loc[m, ID_COLS].assign(prob=prob[m]))
    kept = pd.concat(kept, ignore_index=True) if kept else pd.DataFrame(columns=ID_COLS + ["prob"])

    if exclusive:                       # each S2/S3 record goes to its single most likely S1 entity
        n0 = len(kept)
        kept = keep_best_claimant(kept, thr)
        print(f"exclusive assignment removed {n0 - len(kept)} contested pairs")

    grouped = (kept.groupby("source1_entity_id")["candidate_entity_id"]
               .apply(lambda x: ",".join(sorted(set(x)))).rename("matched_entity_ids").reset_index())

    s1_ids = pd.read_csv(os.path.join(RAW_TEST_DIR, "test_source1.tsv"), sep="\t", usecols=["entity_id"],
                         nrows=sample_size, dtype=str)["entity_id"]
    out = pd.DataFrame({"source1_entity_id": s1_ids}).merge(grouped, on="source1_entity_id", how="left")
    out["matched_entity_ids"] = out["matched_entity_ids"].fillna("")
    out.to_csv(os.path.join(OUTPUT_DIR, "matching_results.tsv"), sep="\t", index=False)

    shutil.copyfile(_candidate_file("test", sample_size), os.path.join(OUTPUT_DIR, "candidate_pairs.tsv"))

    n_match = (out["matched_entity_ids"] != "").sum()
    print(f"matching_results.tsv: {len(out)} S1 entities, {n_match} with >=1 match "
          f"({len(kept)} matched pairs, {len(kept) / max(n_match, 1):.2f} per matched entity)")
    return out


build_features("test", sample_size=SAMPLE_SIZE)          # reuses the vectorizers fitted on train
submission = write_submission(BEST_MODEL, FINAL_THR, SAMPLE_SIZE, USE_EXCLUSIVE)
submission.head()

## 9. Validate the submission (official validator)

In smoke mode the validator is pointed at a truncated copy of `test_source1.tsv` (only the first 5000 S1
entities exist in the smoke submission).

In [ ]:
test_dir = RAW_TEST_DIR
if SAMPLE_SIZE:
    test_dir = tempfile.mkdtemp()
    with open(os.path.join(RAW_TEST_DIR, "test_source1.tsv"), encoding="utf-8") as src, \
         open(os.path.join(test_dir, "test_source1.tsv"), "w", encoding="utf-8") as dst:
        for line_no, line in enumerate(src):
            if line_no > SAMPLE_SIZE:          # header + SAMPLE_SIZE rows
                break
            dst.write(line)

res = subprocess.run(
    [sys.executable, VALIDATOR,
     "--matching", os.path.join(OUTPUT_DIR, "matching_results.tsv"),
     "--candidate", os.path.join(OUTPUT_DIR, "candidate_pairs.tsv"),
     "--test-dir", test_dir],
    capture_output=True, text=True)
print(res.stdout, res.stderr)

## 10. Keep the results (Colab's local disk is wiped when the runtime resets)

In [ ]:
if DATASET_DIR.startswith("/content/drive"):
    print("Dataset, features, model and output already live in Drive under", os.path.dirname(DATASET_DIR.rstrip("/")))
else:
    DRIVE_OUT = os.path.join(DRIVE_ROOT, "outputs")
    for name, d in (("normalized", NORM_DIR), ("candidates", CAND_DIR), ("pairs", PAIRS_DIR),
                    ("features", FEATURES_DIR), ("model", MODEL_DIR), ("output", OUTPUT_DIR)):
        shutil.copytree(d, os.path.join(DRIVE_OUT, name), dirs_exist_ok=True)
    print("Copied to", DRIVE_OUT)